# SmartCampus educational AI - team continuation notebook

This portable edition reads the saved project, without pretending this notebook
was executed on your machine. **Run All makes no API requests with the defaults.**
The student app is `streamlit_app.py`: one uploaded lecture, summary, detailed guide,
quiz, source viewer, tutor and feedback. Use the root README for setup.

All seven test lectures are included. Choose `iot`, `calculus`, `software`,
`big_data`, `cloud`, `data_analysis` or `nlp`; provider is `gemini` or `groq`.
The local model remains deferred to the separate Colab GPU notebook.


In [ ]:
from pathlib import Path
import os, sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'handoff_setup.py').is_file()), None)
if ROOT is None: raise RuntimeError('Open the notebook from the extracted team handoff/repository folder.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from handoff_setup import prepare
PROJECT = prepare()
os.chdir(PROJECT)
if str(PROJECT) not in sys.path: sys.path.insert(0,str(PROJECT))
from notebook_engine import initialize_notebook
SELECTED_LECTURE = globals().get('SELECTED_LECTURE','iot')
PROVIDER = globals().get('PROVIDER','gemini')
CONFIG_PATH = PROJECT/'run/lecture_comparison_v3/configs'/f'{SELECTED_LECTURE}_{PROVIDER}.json'
initialize_notebook(globals())
from math_rendering import math_markdown
from lecture_quality import external_content_findings, quiz_editorial_findings, quiz_for_practice
from provider_availability import current, message

SELECTED_LECTURE='iot'
PROVIDER='gemini'
CONFIG_PATH=PROJECT/'run/lecture_comparison_v3/configs'/f'{SELECTED_LECTURE}_{PROVIDER}.json'
initialize_notebook(globals())
display({'lecture':workflow.doc['name'],'model':workflow.config['model'],'pages':workflow.coverage(),
         'guide_chapters_saved':sum(bool(c.get('parsed')) for c in workflow.state.get('guide_chapters',{}).values())})
quota=current(workflow.config['model'])
if quota: display(Markdown(message(quota)))

## Lecture summary

The short overview is separate from the comprehensive guide. Missing means unfinished, not a zero quality score.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'handoff_setup.py').is_file()), None)
if ROOT is None: raise RuntimeError('Open the notebook from the extracted team handoff/repository folder.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from handoff_setup import prepare
PROJECT = prepare()
os.chdir(PROJECT)
if str(PROJECT) not in sys.path: sys.path.insert(0,str(PROJECT))
from notebook_engine import initialize_notebook
SELECTED_LECTURE = globals().get('SELECTED_LECTURE','iot')
PROVIDER = globals().get('PROVIDER','gemini')
CONFIG_PATH = PROJECT/'run/lecture_comparison_v3/configs'/f'{SELECTED_LECTURE}_{PROVIDER}.json'
initialize_notebook(globals())
from math_rendering import math_markdown
from lecture_quality import external_content_findings, quiz_editorial_findings, quiz_for_practice
from provider_availability import current, message

summary=workflow.state['artifacts'].get('lecture_summary',{})
if summary.get('parsed'): display(Markdown(math_markdown(link_sources(summary['parsed']['body'],workflow))))
else: print('No saved short summary for this lecture/provider yet.')

## Detailed study guide

Automated source checks are screening, not instructor approval. Read any independent source clarifications.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'handoff_setup.py').is_file()), None)
if ROOT is None: raise RuntimeError('Open the notebook from the extracted team handoff/repository folder.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from handoff_setup import prepare
PROJECT = prepare()
os.chdir(PROJECT)
if str(PROJECT) not in sys.path: sys.path.insert(0,str(PROJECT))
from notebook_engine import initialize_notebook
SELECTED_LECTURE = globals().get('SELECTED_LECTURE','iot')
PROVIDER = globals().get('PROVIDER','gemini')
CONFIG_PATH = PROJECT/'run/lecture_comparison_v3/configs'/f'{SELECTED_LECTURE}_{PROVIDER}.json'
initialize_notebook(globals())
from math_rendering import math_markdown
from lecture_quality import external_content_findings, quiz_editorial_findings, quiz_for_practice
from provider_availability import current, message

guide=workflow.state['artifacts'].get('study_guide',{})
display(guide.get('content_review',{}))
for finding in external_content_findings(guide): display(Markdown(finding.get('clarification',finding['reason'])))
if guide.get('parsed'): display(Markdown(math_markdown(link_sources(render_guide(guide),workflow))))
else:
    print('Full guide unfinished. Saved chapters below are a PARTIAL draft.')
    for chapter in workflow.state.get('guide_chapters',{}).values():
        if chapter.get('parsed'): display(Markdown(math_markdown(link_sources(chapter['parsed']['body'],workflow))))

## Quiz and local grading

Flagged quizzes can be inspected but cannot be graded. Difficulty labels are not calibrated.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'handoff_setup.py').is_file()), None)
if ROOT is None: raise RuntimeError('Open the notebook from the extracted team handoff/repository folder.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from handoff_setup import prepare
PROJECT = prepare()
os.chdir(PROJECT)
if str(PROJECT) not in sys.path: sys.path.insert(0,str(PROJECT))
from notebook_engine import initialize_notebook
SELECTED_LECTURE = globals().get('SELECTED_LECTURE','iot')
PROVIDER = globals().get('PROVIDER','gemini')
CONFIG_PATH = PROJECT/'run/lecture_comparison_v3/configs'/f'{SELECTED_LECTURE}_{PROVIDER}.json'
initialize_notebook(globals())
from math_rendering import math_markdown
from lecture_quality import external_content_findings, quiz_editorial_findings, quiz_for_practice
from provider_availability import current, message

record=workflow.state['artifacts'].get('lecture_quiz',{})
quiz=record.get('parsed')
flags=external_content_findings(record)
editorial=[{'item':i,'findings':quiz_editorial_findings(q)} for i,q in enumerate((quiz or {}).get('questions',[]),1)]
display(record.get('content_review',{})); display(flags)
if quiz:
    presented=quiz_for_practice(quiz,{'team_handoff':True,'source':workflow.doc['sha256']})
    show_quiz(presented,workflow)
else: print('No saved quiz yet.')
QUIZ_ANSWERS=[]
SUBMIT_QUIZ=False
if SUBMIT_QUIZ:
    if not quiz or flags or record.get('content_review',{}).get('status')!='passed_automated_review' or any(x['findings'] for x in editorial):
        print('Quiz draft: finish content review before grading.')
    else: show_quiz_feedback(grade_quiz(presented,QUIZ_ANSWERS),workflow)

## Original source

Compare generated claims with the original physical page, not printed slide numbering.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'handoff_setup.py').is_file()), None)
if ROOT is None: raise RuntimeError('Open the notebook from the extracted team handoff/repository folder.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from handoff_setup import prepare
PROJECT = prepare()
os.chdir(PROJECT)
if str(PROJECT) not in sys.path: sys.path.insert(0,str(PROJECT))
from notebook_engine import initialize_notebook
SELECTED_LECTURE = globals().get('SELECTED_LECTURE','iot')
PROVIDER = globals().get('PROVIDER','gemini')
CONFIG_PATH = PROJECT/'run/lecture_comparison_v3/configs'/f'{SELECTED_LECTURE}_{PROVIDER}.json'
initialize_notebook(globals())
from math_rendering import math_markdown
from lecture_quality import external_content_findings, quiz_editorial_findings, quiz_for_practice
from provider_availability import current, message

PAGE=4
page=workflow.doc['pages'][PAGE-1]
display(Image(filename=page['image']))
print(page['text'])

## Tutor - explicit live control

Your own API key and verified free-plan flag are required. The generation and source audit both consume quota.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'handoff_setup.py').is_file()), None)
if ROOT is None: raise RuntimeError('Open the notebook from the extracted team handoff/repository folder.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from handoff_setup import prepare
PROJECT = prepare()
os.chdir(PROJECT)
if str(PROJECT) not in sys.path: sys.path.insert(0,str(PROJECT))
from notebook_engine import initialize_notebook
SELECTED_LECTURE = globals().get('SELECTED_LECTURE','iot')
PROVIDER = globals().get('PROVIDER','gemini')
CONFIG_PATH = PROJECT/'run/lecture_comparison_v3/configs'/f'{SELECTED_LECTURE}_{PROVIDER}.json'
initialize_notebook(globals())
from math_rendering import math_markdown
from lecture_quality import external_content_findings, quiz_editorial_findings, quiz_for_practice
from provider_availability import current, message

QUESTION=''
MODE='direct' # direct, hint, socratic, practice
RUN_TUTOR=False
if RUN_TUTOR and QUESTION.strip():
    response,retrieval=workflow.tutor(QUESTION,MODE,session='team_notebook')
    show_tutor_response(response,retrieval,workflow)
else: print('No live tutor request made.')

## Resume only unfinished work

Saved accepted pages, chapters and request ledgers are reused. Do not reset counters to bypass provider quotas.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'handoff_setup.py').is_file()), None)
if ROOT is None: raise RuntimeError('Open the notebook from the extracted team handoff/repository folder.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from handoff_setup import prepare
PROJECT = prepare()
os.chdir(PROJECT)
if str(PROJECT) not in sys.path: sys.path.insert(0,str(PROJECT))
from notebook_engine import initialize_notebook
SELECTED_LECTURE = globals().get('SELECTED_LECTURE','iot')
PROVIDER = globals().get('PROVIDER','gemini')
CONFIG_PATH = PROJECT/'run/lecture_comparison_v3/configs'/f'{SELECTED_LECTURE}_{PROVIDER}.json'
initialize_notebook(globals())
from math_rendering import math_markdown
from lecture_quality import external_content_findings, quiz_editorial_findings, quiz_for_practice
from provider_availability import current, message

RUN_RESUME=False
if RUN_RESUME:
    from lecture_summary import build_summary
    from comprehensive_guide import build_guide
    from lecture_quiz import build_quiz
    if workflow.process():
        print('Summary:',build_summary(workflow))
        print('Guide:',build_guide(workflow))
        print('Quiz/content checks:',build_quiz(workflow))
else: print('No generation request made.')

## Actual comparison status and evaluation limits

Four RAGAS metric families are distinct from test counts and score values. Concept coverage and quiz validity add checks; missing scores remain missing. References for the newly added four lectures still need preparation. No new instructor ratings or student learning gains are claimed.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'handoff_setup.py').is_file()), None)
if ROOT is None: raise RuntimeError('Open the notebook from the extracted team handoff/repository folder.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from handoff_setup import prepare
PROJECT = prepare()
os.chdir(PROJECT)
if str(PROJECT) not in sys.path: sys.path.insert(0,str(PROJECT))
from notebook_engine import initialize_notebook
SELECTED_LECTURE = globals().get('SELECTED_LECTURE','iot')
PROVIDER = globals().get('PROVIDER','gemini')
CONFIG_PATH = PROJECT/'run/lecture_comparison_v3/configs'/f'{SELECTED_LECTURE}_{PROVIDER}.json'
initialize_notebook(globals())
from math_rendering import math_markdown
from lecture_quality import external_content_findings, quiz_editorial_findings, quiz_for_practice
from provider_availability import current, message

report=PROJECT/'run/lecture_comparison_v3/COMPARISON_BRIEF.md'
if report.exists(): display(Markdown(report.read_text(encoding='utf-8')))
scores=read(PROJECT/'run/lecture_comparison_v3/ragas_scores.json',{})
display([{'lecture':r['lecture'],'arm':r['arm'],'metric':r['metric'],'value':r.get('value'),'status':r['status']} for r in scores.get('rows',[])])
print('Historical local-model comparison is separate: run/comparison_v2/UPGRADE_REPORT.md')